In [7]:
# =========================
# Step 1-1. 기본 라이브러리 로드
# =========================

# 경로 처리를 위한 라이브러리
from pathlib import Path

# 경고 메시지를 줄이기 위한 설정
import warnings
warnings.filterwarnings("ignore")

# 데이터 분석 기본 라이브러리
import numpy as np
import pandas as pd

# 시각화 라이브러리
import matplotlib.pyplot as plt

# 출력 옵션 설정
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# 재현성을 위한 랜덤 시드
RANDOM_STATE = 42

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [8]:
# =========================
# Step 1-2. 프로젝트 경로 설정
# =========================

# 현재 노트북 실행 위치 확인
CURRENT_DIR = Path.cwd()

# VS Code에서 notebooks 폴더 안에서 실행하는 경우와
# 프로젝트 루트에서 실행하는 경우를 모두 처리
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# 주요 폴더 경로 설정
DATA_DIR = PROJECT_ROOT / "data"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

# 필요한 폴더가 없으면 생성
DATA_DIR.mkdir(exist_ok=True)
NOTEBOOK_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

print("현재 실행 위치:", CURRENT_DIR)
print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 폴더:", DATA_DIR)
print("모델 저장 폴더:", MODEL_DIR)
print("리포트 저장 폴더:", REPORT_DIR)

현재 실행 위치: c:\ml-learning
프로젝트 루트: c:\ml-learning
데이터 폴더: c:\ml-learning\data
모델 저장 폴더: c:\ml-learning\models
리포트 저장 폴더: c:\ml-learning\reports


In [9]:
# =========================
# Step 1-3. data 폴더 내 CSV 파일 확인
# =========================

# data 폴더 안의 CSV 파일 목록 확인
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("발견된 CSV 파일 수:", len(csv_files))

for file in csv_files:
    print("-", file.name)

발견된 CSV 파일 수: 1
- Loan_default.csv


In [19]:
# =========================
# Step 1-4. 데이터 로드
# =========================

# CSV 파일이 없는 경우 에러 발생
if len(csv_files) == 0:
    raise FileNotFoundError(
        "data 폴더에 CSV 파일이 없습니다. "
        "Kaggle에서 다운로드한 Loan_default.csv 파일을 data 폴더에 넣어주세요."
    )

# CSV 파일이 여러 개 있으면 첫 번째 파일을 우선 사용
# 이후 파일명이 다르면 직접 data_path를 수정하면 됨
data_path = csv_files[0]

# 데이터 로드
df = pd.read_csv(data_path)

print("사용 데이터 파일:", data_path.name)
print("데이터 크기:", df.shape)

사용 데이터 파일: Loan_default.csv
데이터 크기: (255347, 18)


In [ ]:
# =========================
# Step 1-5. 데이터 샘플 확인
# =========================

# 무작위 10개 행 확인
display(df.sample(10, random_state=RANDOM_STATE))

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
51139,CKV34LU7V7,55,112656,92393,581,113,2,23.54,36,0.15,PhD,Self-employed,Single,Yes,Yes,Home,No,0
71005,62KTYNH93J,56,91569,131575,641,54,1,15.19,12,0.43,High School,Part-time,Divorced,Yes,Yes,Education,Yes,0
35684,JGFUSOIUH7,26,78169,75417,569,105,3,18.02,12,0.29,Master's,Part-time,Married,Yes,Yes,Education,Yes,0
174087,4538THBHOX,26,63033,10804,326,118,1,14.71,24,0.41,High School,Part-time,Single,No,No,Business,Yes,0
137952,DXLNA06JHR,24,29665,21182,662,102,3,15.02,60,0.69,PhD,Unemployed,Single,No,Yes,Business,Yes,0
137678,GPXN95JGCL,41,98511,187008,731,20,4,14.88,36,0.46,Master's,Full-time,Divorced,Yes,No,Other,No,0
74249,FIIMGYJEVG,66,49411,159105,371,86,4,13.35,60,0.12,Bachelor's,Self-employed,Married,No,No,Business,Yes,0
253344,22OCHURLWV,34,117617,173967,518,86,4,2.21,12,0.39,High School,Part-time,Single,No,Yes,Business,Yes,0
219115,OKUSLEJ0SG,67,94473,41189,542,15,2,10.17,36,0.33,Master's,Self-employed,Single,No,Yes,Home,No,0
144898,CL8B73H82O,41,95332,151275,363,39,3,12.88,36,0.66,High School,Self-employed,Single,Yes,Yes,Home,Yes,0


In [13]:
# =========================
# Step 2-1. 컬럼 한글명 매핑표 생성
# =========================

column_description = pd.DataFrame([
    ["LoanID", "대출ID", "식별자", "제외", "각 대출 신청 건을 구분하는 ID"],
    ["Age", "나이", "수치형", "사용", "대출 신청자의 연령"],
    ["Income", "연소득", "수치형", "사용", "대출 신청자의 연간 소득"],
    ["LoanAmount", "대출금액", "수치형", "사용", "신청 또는 실행된 대출 금액"],
    ["CreditScore", "신용점수", "수치형", "사용", "신청자의 신용 평가 점수"],
    ["MonthsEmployed", "근속개월수", "수치형", "사용", "현재 직장 또는 고용 상태 유지 기간"],
    ["NumCreditLines", "신용거래라인수", "수치형", "사용", "보유 중인 신용 계좌 또는 신용 한도 개수"],
    ["InterestRate", "대출금리", "수치형", "사용", "대출에 적용된 이자율"],
    ["LoanTerm", "대출기간", "수치형", "사용", "대출 상환 기간"],
    ["DTIRatio", "부채소득비율", "수치형", "사용", "소득 대비 부채 부담 비율"],
    ["Education", "학력수준", "범주형", "사용", "신청자의 최종 학력"],
    ["EmploymentType", "고용형태", "범주형", "사용", "정규직, 자영업 등 고용 유형"],
    ["MaritalStatus", "결혼상태", "범주형", "사용", "미혼, 기혼 등 혼인 상태"],
    ["HasMortgage", "주택담보대출보유여부", "범주형", "사용", "기존 주택담보대출 보유 여부"],
    ["HasDependents", "부양가족여부", "범주형", "사용", "부양가족 존재 여부"],
    ["LoanPurpose", "대출목적", "범주형", "사용", "대출 사용 목적"],
    ["HasCoSigner", "공동서명자여부", "범주형", "사용", "공동서명자 또는 보증인 존재 여부"],
    ["Default", "부도여부", "타깃", "예측 대상", "대출 부도 여부"],
], columns=["원본 컬럼명", "한글 컬럼명", "변수 유형", "모델 사용 여부", "설명"])

display(column_description)

,원본 컬럼명,한글 컬럼명,변수 유형,모델 사용 여부,설명
0,LoanID,대출ID,식별자,제외,각 대출 신청 건을 구분하는 ID
1,Age,나이,수치형,사용,대출 신청자의 연령
2,Income,연소득,수치형,사용,대출 신청자의 연간 소득
3,LoanAmount,대출금액,수치형,사용,신청 또는 실행된 대출 금액
4,CreditScore,신용점수,수치형,사용,신청자의 신용 평가 점수
5,MonthsEmployed,근속개월수,수치형,사용,현재 직장 또는 고용 상태 유지 기간
6,NumCreditLines,신용거래라인수,수치형,사용,보유 중인 신용 계좌 또는 신용 한도 개수
7,InterestRate,대출금리,수치형,사용,대출에 적용된 이자율
8,LoanTerm,대출기간,수치형,사용,대출 상환 기간
9,DTIRatio,부채소득비율,수치형,사용,소득 대비 부채 부담 비율


In [14]:
# =========================
# Step 2-2. 실제 데이터 컬럼과 매핑표 일치 여부 확인
# =========================

actual_columns = set(df.columns)
mapped_columns = set(column_description["원본 컬럼명"])

missing_in_mapping = actual_columns - mapped_columns
missing_in_data = mapped_columns - actual_columns

print("매핑표에 없는 실제 데이터 컬럼:", missing_in_mapping)
print("실제 데이터에 없는 매핑표 컬럼:", missing_in_data)

display(
    column_description.assign(
        실제데이터존재여부=column_description["원본 컬럼명"].isin(df.columns)
    )
)

매핑표에 없는 실제 데이터 컬럼: set()
실제 데이터에 없는 매핑표 컬럼: set()


,원본 컬럼명,한글 컬럼명,변수 유형,모델 사용 여부,설명,실제데이터존재여부
0,LoanID,대출ID,식별자,제외,각 대출 신청 건을 구분하는 ID,True
1,Age,나이,수치형,사용,대출 신청자의 연령,True
2,Income,연소득,수치형,사용,대출 신청자의 연간 소득,True
3,LoanAmount,대출금액,수치형,사용,신청 또는 실행된 대출 금액,True
4,CreditScore,신용점수,수치형,사용,신청자의 신용 평가 점수,True
5,MonthsEmployed,근속개월수,수치형,사용,현재 직장 또는 고용 상태 유지 기간,True
6,NumCreditLines,신용거래라인수,수치형,사용,보유 중인 신용 계좌 또는 신용 한도 개수,True
7,InterestRate,대출금리,수치형,사용,대출에 적용된 이자율,True
8,LoanTerm,대출기간,수치형,사용,대출 상환 기간,True
9,DTIRatio,부채소득비율,수치형,사용,소득 대비 부채 부담 비율,True


In [15]:
# =========================
# Step 2-3. 한글 컬럼명 데이터프레임 생성
# =========================

column_rename_dict = dict(
    zip(
        column_description["원본 컬럼명"],
        column_description["한글 컬럼명"]
    )
)

df_kr = df.rename(columns=column_rename_dict)

display(df_kr.head())
display(df_kr.info())

,대출ID,나이,연소득,대출금액,신용점수,근속개월수,신용거래라인수,대출금리,대출기간,부채소득비율,학력수준,고용형태,결혼상태,주택담보대출보유여부,부양가족여부,대출목적,공동서명자여부,부도여부
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 18 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   대출ID        255347 non-null  object 
 1   나이          255347 non-null  int64  
 2   연소득         255347 non-null  int64  
 3   대출금액        255347 non-null  int64  
 4   신용점수        255347 non-null  int64  
 5   근속개월수       255347 non-null  int64  
 6   신용거래라인수     255347 non-null  int64  
 7   대출금리        255347 non-null  float64
 8   대출기간        255347 non-null  int64  
 9   부채소득비율      255347 non-null  float64
 10  학력수준        255347 non-null  object 
 11  고용형태        255347 non-null  object 
 12  결혼상태        255347 non-null  object 
 13  주택담보대출보유여부  255347 non-null  object 
 14  부양가족여부      255347 non-null  object 
 15  대출목적        255347 non-null  object 
 16  공동서명자여부     255347 non-null  object 
 17  부도여부        255347 non-null  int64  
dtypes: float64(2), int64(8), object(8)
memory us

None

In [ ]:
# =========================
# Step 2-4. 한글 컬럼명 데이터셋 CSV 저장
# =========================

# 원본 df는 그대로 두고, 한글 컬럼명 복사본만 저장
df_kr_path = REPORT_DIR / "loan_default_korean_columns.csv"

df_kr.to_csv(
    df_kr_path,
    index=False,
    encoding="utf-8-sig"
)

print("한글 컬럼명 데이터셋 CSV 저장 완료")
print("저장 위치:", df_kr_path)

# 저장 결과 일부 확인
df_kr_check = pd.read_csv(df_kr_path, encoding="utf-8-sig", nrows=5)
display(df_kr_check)

한글 컬럼명 데이터셋 CSV 저장 완료
저장 위치: c:\ml-learning\reports\loan_default_korean_columns.csv


,대출ID,나이,연소득,대출금액,신용점수,근속개월수,신용거래라인수,대출금리,대출기간,부채소득비율,학력수준,고용형태,결혼상태,주택담보대출보유여부,부양가족여부,대출목적,공동서명자여부,부도여부
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [18]:
# =========================
# Step 3-1. 컬럼별 기본 품질 점검
# =========================

# 컬럼별 자료형, 고유값 수, 결측치 수, 결측치 비율 확인
data_quality = pd.DataFrame({
    "원본컬럼명": df.columns,
    "자료형": df.dtypes.astype(str).values,
    "고유값수": df.nunique(dropna=False).values,
    "결측치수": df.isna().sum().values,
    "결측치비율": df.isna().mean().values
})

# Step 2에서 만든 column_description이 있으면 한글 컬럼명도 붙임
if "column_description" in globals():
    data_quality = data_quality.merge(
        column_description[["원본 컬럼명", "한글 컬럼명", "변수 유형", "모델 사용 여부"]],
        left_on="원본컬럼명",
        right_on="원본 컬럼명",
        how="left"
    ).drop(columns=["원본 컬럼명"])

    data_quality = data_quality[
        ["원본컬럼명", "한글 컬럼명", "변수 유형", "모델 사용 여부", "자료형", "고유값수", "결측치수", "결측치비율"]
    ]

display(data_quality)

,원본컬럼명,한글 컬럼명,변수 유형,모델 사용 여부,자료형,고유값수,결측치수,결측치비율
0,LoanID,대출ID,식별자,제외,object,255347,0,0.0
1,Age,나이,수치형,사용,int64,52,0,0.0
2,Income,연소득,수치형,사용,int64,114620,0,0.0
3,LoanAmount,대출금액,수치형,사용,int64,158729,0,0.0
4,CreditScore,신용점수,수치형,사용,int64,550,0,0.0
5,MonthsEmployed,근속개월수,수치형,사용,int64,120,0,0.0
6,NumCreditLines,신용거래라인수,수치형,사용,int64,4,0,0.0
7,InterestRate,대출금리,수치형,사용,float64,2301,0,0.0
8,LoanTerm,대출기간,수치형,사용,int64,5,0,0.0
9,DTIRatio,부채소득비율,수치형,사용,float64,81,0,0.0


In [22]:
# =========================
# Step 3-2. 중복 행 확인
# =========================

# 전체 행 기준 중복 여부 확인
duplicated_count = df.duplicated().sum()
duplicated_ratio = duplicated_count / len(df)

print("전체 중복 행 수:", duplicated_count)
print("전체 중복 행 비율:", round(duplicated_ratio, 6))

# 중복 행이 있다면 일부 확인
if duplicated_count > 0:
    display(df[df.duplicated(keep=False)].head(20))
else:
    print("전체 행 기준 중복 데이터는 없습니다.")

전체 중복 행 수: 0
전체 중복 행 비율: 0.0
전체 행 기준 중복 데이터는 없습니다.


In [23]:
# =========================
# Step 3-3. ID 컬럼 중복 확인
# =========================

# ID처럼 보이는 컬럼 자동 탐색
id_like_cols = [
    col for col in df.columns
    if col.lower() in ["id", "loanid", "loan_id", "customerid", "customer_id"]
    or col.lower().endswith("id")
]

print("ID 후보 컬럼:", id_like_cols)

# ID 후보 컬럼별 중복 확인
for col in id_like_cols:
    duplicate_id_count = df[col].duplicated().sum()
    unique_id_count = df[col].nunique(dropna=False)

    print()
    print(f"[{col}]")
    print("고유 ID 수:", unique_id_count)
    print("중복 ID 수:", duplicate_id_count)
    print("전체 행 수:", len(df))

    if duplicate_id_count > 0:
        display(df[df[col].duplicated(keep=False)].sort_values(col).head(20))

ID 후보 컬럼: ['LoanID']

[LoanID]
고유 ID 수: 255347
중복 ID 수: 0
전체 행 수: 255347


In [24]:
# =========================
# Step 3-4. 타깃 변수 상태 확인
# =========================

# 기본 타깃 컬럼명
TARGET_COL = "Default"

# 타깃 컬럼 존재 여부 확인
if TARGET_COL not in df.columns:
    raise ValueError(f"{TARGET_COL} 컬럼이 없습니다. 실제 타깃 컬럼명을 확인하세요.")

# 타깃 값 분포 확인
target_count = df[TARGET_COL].value_counts(dropna=False).sort_index()
target_ratio = df[TARGET_COL].value_counts(normalize=True, dropna=False).sort_index()

target_summary = pd.DataFrame({
    "count": target_count,
    "ratio": target_ratio
})

display(target_summary)

print("타깃 결측치 수:", df[TARGET_COL].isna().sum())
print("타깃 고유값:", sorted(df[TARGET_COL].dropna().unique()))

,count,ratio
Default,,
0,225694,0.883872
1,29653,0.116128


타깃 결측치 수: 0
타깃 고유값: [0, 1]


In [25]:
# =========================
# Step 3-5. 수치형·범주형 컬럼 자동 분리
# =========================

# 모델에 바로 쓰지 않을 컬럼
exclude_cols = [TARGET_COL] + id_like_cols

# 입력 변수 후보
feature_cols = [col for col in df.columns if col not in exclude_cols]

# 수치형 변수
numeric_cols = df[feature_cols].select_dtypes(include=["number"]).columns.tolist()

# 범주형 변수
categorical_cols = df[feature_cols].select_dtypes(exclude=["number"]).columns.tolist()

print("제외 컬럼:", exclude_cols)
print()
print("입력 변수 수:", len(feature_cols))
print("수치형 변수 수:", len(numeric_cols))
print(numeric_cols)
print()
print("범주형 변수 수:", len(categorical_cols))
print(categorical_cols)

제외 컬럼: ['Default', 'LoanID']

입력 변수 수: 16
수치형 변수 수: 9
['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio']

범주형 변수 수: 7
['Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']


In [26]:
# =========================
# Step 3-6. 범주형 변수 값 확인
# =========================

# 범주형 변수의 고유값과 빈도 확인
for col in categorical_cols:
    print("=" * 80)
    print(f"[{col}] 고유값 수:", df[col].nunique(dropna=False))
    display(df[col].value_counts(dropna=False).to_frame("count"))

[Education] 고유값 수: 4


,count
Education,
Bachelor's,64366
High School,63903
Master's,63541
PhD,63537


[EmploymentType] 고유값 수: 4


,count
EmploymentType,
Part-time,64161
Unemployed,63824
Self-employed,63706
Full-time,63656


[MaritalStatus] 고유값 수: 3


,count
MaritalStatus,
Married,85302
Divorced,85033
Single,85012


... (4 outputs left out)

[LoanPurpose] 고유값 수: 5


,count
LoanPurpose,
Business,51298
Home,51286
Education,51005
Other,50914
Auto,50844


[HasCoSigner] 고유값 수: 2


,count
HasCoSigner,
Yes,127701
No,127646


In [27]:
# =========================
# Step 3-7. 수치형 변수 이상 범위 1차 확인
# =========================

# 수치형 변수의 최솟값, 최댓값, 평균, 중앙값 확인
numeric_range_check = df[numeric_cols].agg(["min", "max", "mean", "median"]).T
numeric_range_check["missing_count"] = df[numeric_cols].isna().sum()
numeric_range_check["missing_ratio"] = df[numeric_cols].isna().mean()

display(numeric_range_check)

,min,max,mean,median,missing_count,missing_ratio
Age,18.0,69.0,43.498306,43.00,0,0.0
Income,15000.0,149999.0,82499.304597,82466.00,0,0.0
LoanAmount,5000.0,249999.0,127578.865512,127556.00,0,0.0
CreditScore,300.0,849.0,574.264346,574.00,0,0.0
MonthsEmployed,0.0,119.0,59.541976,60.00,0,0.0
NumCreditLines,1.0,4.0,2.501036,2.00,0,0.0
InterestRate,2.0,25.0,13.492773,13.46,0,0.0
LoanTerm,12.0,60.0,36.025894,36.00,0,0.0
DTIRatio,0.1,0.9,0.500212,0.50,0,0.0


In [28]:
# =========================
# Step 3-8. 데이터 품질 점검 요약
# =========================

summary = {
    "전체 행 수": len(df),
    "전체 컬럼 수": df.shape[1],
    "중복 행 수": int(df.duplicated().sum()),
    "결측치 있는 컬럼 수": int((df.isna().sum() > 0).sum()),
    "입력 변수 수": len(feature_cols),
    "수치형 변수 수": len(numeric_cols),
    "범주형 변수 수": len(categorical_cols),
    "타깃 컬럼": TARGET_COL,
    "타깃 1 비율": float(df[TARGET_COL].mean())
}

summary_df = pd.DataFrame(summary.items(), columns=["항목", "값"])
display(summary_df)

,항목,값
0,전체 행 수,255347
1,전체 컬럼 수,18
2,중복 행 수,0
3,결측치 있는 컬럼 수,0
4,입력 변수 수,16
5,수치형 변수 수,9
6,범주형 변수 수,7
7,타깃 컬럼,Default
8,타깃 1 비율,0.116128
